# EDA du Dataset BTGenBot
Analyse exploratoire du dataset XML BehaviorTree.CPP pour préparer le fine-tuning.

In [37]:
import json
import xml.etree.ElementTree as ET
from collections import Counter
import numpy as np

In [38]:
# Configuration centrale du notebook
DATASET_RAW_PATH = 'bt_dataset.json'
DATASET_FILTERED_PATH = 'bt_dataset_filtered.json'
DATASET_CHAT_PATH = 'bt_dataset_chat.jsonl'

TOP_K_VOCAB = 15
TOP_K_SKILLS = 25
TOP_K_PATTERNS = 10

SYSTEM_PROMPT_DEFAULT = (
    "You are an expert in robotics and BehaviorTree.CPP. "
    "Your objective is to express the behavior tree task in XML format."
)

In [39]:
# Fonctions utilitaires partagées pour éviter les répétitions XML

XML_DECLARATION = '<?xml version="1.0"?>'
STRUCTURAL_NOISE_TAGS = {
    'dummy_root', 'root', 'BehaviorTree', 'TreeNodesModel',
    'Action', 'Condition', 'Control', 'Decorator',
    'input_port', 'output_port', 'inout_port'
}

def load_json_file(file_path: str):
    """Charge un fichier JSON avec encodage UTF-8."""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def strip_xml_declaration(xml_text: str) -> str:
    """Supprime la déclaration XML standard et les espaces inutiles."""
    return xml_text.replace(XML_DECLARATION, '').strip()

def parse_xml_fragment(xml_text: str, wrap_dummy: bool = True) -> ET.Element:
    """Parse un fragment XML BT, optionnellement encapsulé dans dummy_root."""
    cleaned = strip_xml_declaration(xml_text)
    if wrap_dummy:
        return ET.fromstring(f"<dummy_root>{cleaned}</dummy_root>")
    return ET.fromstring(cleaned)

def xml_valide_pour_eda(xml_text: str) -> bool:
    """Vérifie qu'un XML BT est parseable pour l'EDA."""
    try:
        parse_xml_fragment(xml_text, wrap_dummy=True)
        return True
    except ET.ParseError:
        return False

def remove_treenodesmodel(root: ET.Element) -> ET.Element:
    """Retourne une copie de l'arbre sans les sous-arbres TreeNodesModel."""
    root_copy = ET.fromstring(ET.tostring(root, encoding='unicode'))
    for parent in root_copy.iter():
        for child in list(parent):
            if child.tag == 'TreeNodesModel':
                parent.remove(child)
    return root_copy

## 1. Qualité et Statistiques du Dataset
### 1.1 Occurrences des nœuds XML

In [40]:
# Charger le dataset complet
dataset = load_json_file(DATASET_RAW_PATH)

vocabulaire_noeuds = Counter()
longueurs_prompt_mots = []
longueurs_xml_mots = []
parse_errors = 0
exemples_parse_errors = []

for idx, item in enumerate(dataset):
    prompt = item.get('input', '')
    xml_text = item.get('output', '')
    longueurs_prompt_mots.append(len(prompt.split()))
    longueurs_xml_mots.append(len(xml_text.split()))

    try:
        root = parse_xml_fragment(xml_text, wrap_dummy=True)
        for elem in root.iter():
            if elem.tag not in ['dummy_root', 'root', 'BehaviorTree']:
                vocabulaire_noeuds[elem.tag] += 1
    except ET.ParseError as e:
        parse_errors += 1
        if len(exemples_parse_errors) < 5:
            exemples_parse_errors.append((idx, str(e)))

print("___ Statistiques du Dataset BTGenBot ___")
print(f"Nombre d'exemples : {len(dataset)}")
print(f"Longueur moyenne Input (mots) : {np.mean(longueurs_prompt_mots):.0f}")
print(f"Longueur moyenne XML (mots) : {np.mean(longueurs_xml_mots):.0f}")
print(f"Exemples XML non parseables : {parse_errors}")
if exemples_parse_errors:
    print("Exemples d'erreurs XML :")
    for idx, err in exemples_parse_errors:
        print(f"- index {idx}: {err}")

print(f"\nVocabulaire des Nœuds (Top {TOP_K_VOCAB}) :")
for noeud, freq in vocabulaire_noeuds.most_common(TOP_K_VOCAB):
    print(f"- {noeud} : {freq} occurrences")

___ Statistiques du Dataset BTGenBot ___
Nombre d'exemples : 594
Longueur moyenne Input (mots) : 115
Longueur moyenne XML (mots) : 141
Exemples XML non parseables : 23
Exemples d'erreurs XML :
- index 5: not well-formed (invalid token): line 2, column 10
- index 6: not well-formed (invalid token): line 2, column 10
- index 47: not well-formed (invalid token): line 2, column 10
- index 51: XML or text declaration not at start of entity: line 1, column 12
- index 61: XML or text declaration not at start of entity: line 1, column 12

Vocabulaire des Nœuds (Top 15) :
- Action : 5458 occurrences
- input_port : 3133 occurrences
- Condition : 1817 occurrences
- Sequence : 1444 occurrences
- output_port : 897 occurrences
- SubTree : 763 occurrences
- Fallback : 645 occurrences
- SetBlackboard : 361 occurrences
- SequenceStar : 327 occurrences
- inout_port : 308 occurrences
- TreeNodesModel : 225 occurrences
- ClearEntireCostmap : 174 occurrences
- ForceSuccess : 162 occurrences
- TextToSpeechA

### 1.2 Nettoyage XML
23 exemples contiennent un XML invalide et sont filtrés du dataset principal.

In [41]:
# Filtrer les exemples avec XML invalide et sauvegarder un dataset nettoyé
dataset_filtre = [
    item for item in dataset
    if xml_valide_pour_eda(item.get('output', ''))
]

indices_supprimes = [
    idx for idx, item in enumerate(dataset)
    if not xml_valide_pour_eda(item.get('output', ''))
]

with open(DATASET_FILTERED_PATH, 'w', encoding='utf-8') as f:
    json.dump(dataset_filtre, f, ensure_ascii=False, indent=2)

print('___ Filtrage XML du dataset ___')
print(f"Taille originale : {len(dataset)}")
print(f"Taille filtrée : {len(dataset_filtre)}")
print(f"Exemples retirés : {len(indices_supprimes)}")
print(f"Fichier écrit : {DATASET_FILTERED_PATH}")
print("Aperçu indices retirés (10 max) :", indices_supprimes[:10])

___ Filtrage XML du dataset ___
Taille originale : 594
Taille filtrée : 571
Exemples retirés : 23
Fichier écrit : bt_dataset_filtered.json
Aperçu indices retirés (10 max) : [5, 6, 47, 51, 61, 89, 95, 98, 118, 211]


### 1.3 Nœuds les plus communs
Présence des tags à travers les XML valides (fréquence documentaire).

In [42]:
# Nœuds présents au moins une fois dans tous les XML valides
dataset_source = dataset_filtre if 'dataset_filtre' in globals() else [
    item for item in dataset if xml_valide_pour_eda(item.get('output', ''))
]

doc_freq = Counter()
noeuds_communs = None

for item in dataset_source:
    root = parse_xml_fragment(item.get('output', ''), wrap_dummy=True)

    # Ensemble des tags présents dans cet XML (présence/absence, pas le nombre d'occurrences)
    tags_xml = {
        elem.tag for elem in root.iter()
        if elem.tag not in ['dummy_root', 'root']
    }

    for tag in tags_xml:
        doc_freq[tag] += 1

    if noeuds_communs is None:
        noeuds_communs = set(tags_xml)
    else:
        noeuds_communs &= tags_xml

noeuds_communs = sorted(noeuds_communs) if noeuds_communs else []
n_xml = len(dataset_source)

print('___ Nœuds présents dans 100% des XML valides ___')
print(f"Nombre de XML considérés : {n_xml}")
print(f"Nombre de nœuds communs : {len(noeuds_communs)}")
for tag in noeuds_communs:
    print(f"- {tag}")

print('\n___ Couverture par fichier (Top 15) ___')
for tag, c in doc_freq.most_common(15):
    print(f"- {tag}: {c}/{n_xml} ({100*c/n_xml:.2f}%)")

___ Nœuds présents dans 100% des XML valides ___
Nombre de XML considérés : 571
Nombre de nœuds communs : 0

___ Couverture par fichier (Top 15) ___
- BehaviorTree: 570/571 (99.82%)
- Sequence: 438/571 (76.71%)
- Action: 240/571 (42.03%)
- TreeNodesModel: 225/571 (39.40%)
- Fallback: 188/571 (32.92%)
- Condition: 145/571 (25.39%)
- input_port: 137/571 (23.99%)
- FollowPath: 132/571 (23.12%)
- ComputePathToPose: 128/571 (22.42%)
- SubTree: 124/571 (21.72%)
- Repeat: 119/571 (20.84%)
- PipelineSequence: 106/571 (18.56%)
- output_port: 106/571 (18.56%)
- SequenceStar: 105/571 (18.39%)
- RateController: 99/571 (17.34%)


### Mini-conclusion Section 1
- Le dataset contient une base riche mais hétérogène, avec une variabilité forte des structures XML.
- Le nettoyage retire 23 exemples invalides et fournit un jeu de travail plus fiable pour l'analyse.
- Les nœuds les plus fréquents confirment une forte présence des structures BT standards et des actions de navigation.

## 2. Skills Métier et Classification

### 2.1 Extraction des skills métier
On retire les nœuds standards BehaviorTree.CPP pour isoler les actions métier spécifiques au domaine.

In [43]:
# 1. Liste des nœuds standards, décorateurs et structurels
btcpp_standard_nodes = {
    # Nœuds de contrôle et décorateurs standards
    'BehaviorTree', 'Sequence', 'SequenceStar', 'Fallback', 'FallbackStar',
    'ReactiveSequence', 'ReactiveFallback', 'Parallel', 'PipelineSequence',
    'RecoveryNode', 'RoundRobin', 'Inverter', 'RetryUntilSuccessful',
    'RetryUntilSuccesful', 'KeepRunningUntilFailure', 'ForceSuccess',
    'ForceFailure', 'AlwaysSuccess', 'AlwaysFailure', 'Timeout', 'Delay',
    'RateController', 'DistanceController', 'SpeedController', 'SubTree', 'SubTreePlus',
    'Repeat', 'IfThenElse', 'WhileDo',
    # Blackboard
    'SetBlackboard', 'BlackboardCheckInt', 'BlackboardCheckDouble',
    'BlackboardCheckString', 'BlackboardCheckBool', 'BB_Precondition',
    # Structure XML
    'root', 'dummy_root', 'TreeNodesModel', 'Action', 'Condition',
    'Control', 'Decorator', 'input_port', 'output_port', 'inout_port'
}

skills_potentiels = Counter()

for item in dataset_source:
    xml_text = item.get('output', '')
    try:
        root = parse_xml_fragment(xml_text, wrap_dummy=True)
        root_clean = remove_treenodesmodel(root)

        for elem in root_clean.iter():
            if elem.tag not in btcpp_standard_nodes:
                skills_potentiels[elem.tag] += 1
    except ET.ParseError:
        continue

print('\n___ Liste des Skills Potentiels (Métier) nettoyée ___')
print(f"Nombre total de skills uniques identifiés : {len(skills_potentiels)}")
print(f"Top {TOP_K_SKILLS} des skills les plus fréquents :")
for skill, freq in skills_potentiels.most_common(TOP_K_SKILLS):
    print(f"- {skill} : {freq} occurrences")


___ Liste des Skills Potentiels (Métier) nettoyée ___
Nombre total de skills uniques identifiés : 314
Top 25 des skills les plus fréquents :
- ClearEntireCostmap : 174 occurrences
- TextToSpeechActionClient : 156 occurrences
- ComputePathToPose : 143 occurrences
- FollowPath : 136 occurrences
- GoalUpdated : 83 occurrences
- Wait : 76 occurrences
- MoveAhead : 72 occurrences
- TrackAction : 64 occurrences
- Spin : 61 occurrences
- Turn : 41 occurrences
- MoveBase : 37 occurrences
- AntennaAction : 33 occurrences
- Nav2Client : 30 occurrences
- PlanManipulatorPath : 29 occurrences
- MoveManipulator : 29 occurrences
- RobotSpin : 29 occurrences
- TextCompareAction : 27 occurrences
- NavigateToWp : 25 occurrences
- BaseMovement : 24 occurrences
- HumanPoseDetect : 23 occurrences
- SmileAction : 21 occurrences
- GoalUpdater : 21 occurrences
- CheckBlackboard : 20 occurrences
- Print : 20 occurrences
- ChangeParam : 20 occurrences


### 2.2 Classification des BT par type de skills

In [44]:
def categoriser_bt_ameliore(xml_text: str) -> str:
    """Catégorise le BT en analysant la présence de tags métiers spécifiques."""
    try:
        root = parse_xml_fragment(xml_text, wrap_dummy=True)
        tags = {elem.tag for elem in root.iter()}
    except ET.ParseError:
        return "Erreur Parsing"

    # 1. Manipulation et Bras Robotique
    manipulation_skills = {'PlanManipulatorPath', 'MoveManipulator'}
    if tags.intersection(manipulation_skills):
        if tags.intersection({'ComputePathToPose', 'NavigateToWp', 'MoveBase'}):
            return "Navigation + Manipulation"
        return "Manipulation / Bras Robotique"

    # 2. Vision Active et Interaction
    vision_interaction_skills = {'HumanPoseDetect', 'SmileAction', 'TextToSpeechActionClient'}
    if tags.intersection(vision_interaction_skills):
        return "Vision Active / Interaction Humain-Robot"

    # 3. Navigation
    nav_skills = {'ComputePathToPose', 'FollowPath', 'MoveBase', 'MoveAhead', 'NavigateToWp', 'Nav2Client', 'BaseMovement'}
    if tags.intersection(nav_skills):
        recovery_skills = {'ClearEntireCostmap', 'Spin', 'Wait', 'Turn', 'RobotSpin'}
        if tags.intersection(recovery_skills):
            return "Navigation avec Recovery/Fallback"
        return "Navigation Standard"

    # 4. Tracking et cas particuliers
    if 'TrackAction' in tags or 'AntennaAction' in tags:
        return "Tracking / Action Spécifique"

    return "Autre / Non classifié"

# Application sur le dataset propre
distribution_taches = Counter()
for item in dataset_source:
    cat = categoriser_bt_ameliore(item.get('output', ''))
    distribution_taches[cat] += 1

print('\n___ Distribution des Tâches (Heuristique affinée) ___')
for cat, freq in distribution_taches.most_common():
    print(f"- {cat} : {freq} exemples")


___ Distribution des Tâches (Heuristique affinée) ___
- Autre / Non classifié : 352 exemples
- Navigation Standard : 118 exemples
- Navigation avec Recovery/Fallback : 67 exemples
- Vision Active / Interaction Humain-Robot : 30 exemples
- Manipulation / Bras Robotique : 2 exemples
- Tracking / Action Spécifique : 2 exemples


### Mini-conclusion Section 2
- L'extraction isole un vocabulaire métier large (skills non standards BT.CPP).
- La distribution des catégories montre une dominance des scénarios de navigation.

## 3. Complexité Structurelle des Arbres

### 3.1 Analyse croisée structure vs catégorie métier

In [45]:
import re
import math
import pandas as pd

# PARTIE 1 : FONCTIONS DE NETTOYAGE

def sanitize_xml(xml_text: str) -> str:
    """Nettoyage minimal pour les défauts XML récurrents sans changer la sémantique."""
    cleaned = xml_text.strip()
    cleaned = re.sub(r"^\ufeff", "", cleaned)
    cleaned = re.sub(
        r"<\?xml\s+version\s*=\s*['\"]1\.0['\"]\s*\?>",
        XML_DECLARATION,
        cleaned
    )
    return cleaned

def parse_xml_resilient(xml_text: str):
    """Parse un XML BT avec fallback sur nettoyage léger."""
    try:
        return parse_xml_fragment(xml_text, wrap_dummy=True), True
    except ET.ParseError:
        try:
            return parse_xml_fragment(sanitize_xml(xml_text), wrap_dummy=True), True
        except ET.ParseError:
            return None, False

def iter_nodes(root: ET.Element) -> list:
    """Parcourt l'arbre XML pour extraire la profondeur de chaque noeud."""
    rows = []
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        rows.append((node, depth))
        for child in reversed(list(node)):
            stack.append((child, depth + 1))
    return rows

def get_tree_metrics(record_id: int, root: ET.Element) -> dict:
    """Calcule la complexité structurelle (Mourad)."""
    nodes = iter_nodes(root)
    depths = [depth for _, depth in nodes]
    child_counts = [len(list(element)) for element, _ in nodes]

    n_nodes = len(nodes)
    max_depth = max(depths) if depths else 0
    internal_counts = [count for count in child_counts if count > 0]
    avg_branching = float(np.mean(internal_counts)) if internal_counts else 0.0
    complexity = n_nodes * math.log2(max_depth + 2) * (avg_branching + 1)

    return {
        "record_id": record_id,
        "n_nodes": n_nodes,
        "max_depth": max_depth,
        "complexity_score": complexity
    }

# PARTIE 2 : CLASSIFICATION MÉTIER

btcpp_standard_nodes_v4 = btcpp_standard_nodes.union({
    'Switch2', 'Switch3', 'Switch4', 'Switch5', 'Switch6', 'include'
})

def get_category_from_root(root: ET.Element) -> str:
    """Heuristique V4 appliquée directement sur l'arbre parsé."""
    root_clean = remove_treenodesmodel(root)
    tags = {elem.tag for elem in root_clean.iter() if elem.tag not in btcpp_standard_nodes_v4}

    manipulation_skills = {'PlanManipulatorPath', 'MoveManipulator', 'OpenGripper', 'CloseGripper', 'ApproachObject'}
    if tags.intersection(manipulation_skills):
        if tags.intersection({'ComputePathToPose', 'NavigateToWp', 'MoveBase', 'NavigateToPose', 'Move'}):
            return "Navigation + Manipulation"
        return "Manipulation / Saisie"

    vision_interaction_skills = {'HumanPoseDetect', 'SmileAction', 'TextToSpeechActionClient'}
    if tags.intersection(vision_interaction_skills):
        return "Vision Active / Interaction Humain-Robot"

    nav_skills = {'ComputePathToPose', 'FollowPath', 'MoveBase', 'MoveAhead', 'NavigateToWp', 'Nav2Client', 'BaseMovement', 'NavigateToPose', 'Move'}
    if tags.intersection(nav_skills):
        recovery_skills = {'ClearEntireCostmap', 'Spin', 'Wait', 'Turn', 'RobotSpin'}
        if tags.intersection(recovery_skills):
            return "Navigation avec Recovery/Fallback"
        if 'AllGoalsAchieved' in tags or 'GetNextGoal' in tags:
            return "Navigation Multi-Waypoints"
        return "Navigation Standard"

    if 'CheckBattery' in tags:
        return "Surveillance / Gestion Batterie"

    if 'TrackAction' in tags or 'AntennaAction' in tags:
        return "Tracking / Action Spécifique"

    logique_skills = {'GoalUpdated', 'GoalUpdater', 'TextCompareAction', 'CheckBlackboard', 'Print', 'ChangeParam'}
    if tags.intersection(logique_skills):
        return "Logique Interne / Configuration"

    if tags.intersection({'Wait', 'Turn'}):
        return "Mouvements Basiques (sans navigation)"

    return "Autre / Non classifié"

# PARTIE 3 : EXÉCUTION DU PIPELINE

print("Chargement du dataset...")
dataset = load_json_file(DATASET_FILTERED_PATH)

metric_rows = []
dataset_for_jsonl = []

for idx, item in enumerate(dataset):
    root, is_valid = parse_xml_resilient(item.get('output', ''))
    if not is_valid:
        continue

    metrics = get_tree_metrics(idx, root)
    metrics['task_category'] = get_category_from_root(root)
    metric_rows.append(metrics)
    dataset_for_jsonl.append(item)

df_metrics = pd.DataFrame(metric_rows)
stats = (
    df_metrics.groupby('task_category')
    .agg(
        nombre_arbres=('record_id', 'count'),
        profondeur_moyenne=('max_depth', 'mean'),
        noeuds_moyens=('n_nodes', 'mean'),
        complexite=('complexity_score', 'mean')
    )
    .sort_values('nombre_arbres', ascending=False)
    .round(1)
)

print("\n___ Analyse Croisée : Structure vs Métier ___")
display(stats)

Chargement du dataset...

___ Analyse Croisée : Structure vs Métier ___


,nombre_arbres,profondeur_moyenne,noeuds_moyens,complexite
task_category,,,,
Autre / Non classifié,301,6.3,54.5,726.1
Navigation Standard,128,5.9,14.3,130.6
Navigation avec Recovery/Fallback,70,7.7,26.6,278.7
Vision Active / Interaction Humain-Robot,30,8.7,38.3,470.6
Manipulation / Saisie,19,4.6,16.2,152.3
Navigation Multi-Waypoints,6,6.3,10.7,82.6
Logique Interne / Configuration,5,4.0,9.2,86.6
Mouvements Basiques (sans navigation),5,4.8,7.6,51.5
Surveillance / Gestion Batterie,4,6.5,39.2,632.0


### Mini-conclusion Section 3
- Les catégories métier n'ont pas la même complexité structurelle (profondeur, nombre de nœuds, score).
- Les scénarios de recovery et interaction présentent des arbres plus profonds en moyenne.

## 4. Motifs de Recovery (Design Patterns)

### 4.1 Extraction des patterns métier
Identification des signatures de sous-arbres les plus fréquentes pour les logiques de recovery.

In [46]:
def get_node_signature(element: ET.Element, current_depth: int = 0, max_depth: int = 2) -> str:
    """
    Génère une signature textuelle d'un sous-arbre (ex: Fallback(Sequence, Sequence(Spin))).
    Ignore le bruit (TreeNodesModel, etc.).
    """
    if current_depth >= max_depth or len(list(element)) == 0:
        return element.tag

    children_signatures = []
    for child in element:
        if child.tag not in STRUCTURAL_NOISE_TAGS:
            children_signatures.append(get_node_signature(child, current_depth + 1, max_depth))

    if not children_signatures:
        return element.tag

    return f"{element.tag}({', '.join(children_signatures)})"

dataset = load_json_file(DATASET_FILTERED_PATH)

patterns_fallback = Counter()
patterns_globaux = Counter()

for item in dataset:
    try:
        root = parse_xml_fragment(item.get('output', ''), wrap_dummy=True)
        root_clean = remove_treenodesmodel(root)

        for elem in root_clean.iter():
            if elem.tag in STRUCTURAL_NOISE_TAGS:
                continue

            signature = get_node_signature(elem, max_depth=2)
            patterns_globaux[signature] += 1

            if elem.tag in ['Fallback', 'ReactiveFallback', 'RecoveryNode']:
                patterns_fallback[signature] += 1
    except ET.ParseError:
        continue

print(f"___ Top {TOP_K_PATTERNS} des Design Patterns de Recovery (Fallback) ___")
df_fallbacks = pd.DataFrame(
    patterns_fallback.most_common(TOP_K_PATTERNS),
    columns=['Pattern (Signature)', 'Occurrences']
 )
display(df_fallbacks)

print(f"\n___ Top {TOP_K_PATTERNS} des Design Patterns Globaux (Toutes catégories) ___")
df_globaux = pd.DataFrame(
    patterns_globaux.most_common(TOP_K_PATTERNS),
    columns=['Pattern (Signature)', 'Occurrences']
 )
display(df_globaux)

___ Top 10 des Design Patterns de Recovery (Fallback) ___


,Pattern (Signature),Occurrences
0,Fallback,85
1,Fallback(ForceFailure),52
2,Fallback(Sequence),45
3,"Fallback(Sequence, Sequence)",28
4,"RecoveryNode(FollowPath, ClearEntireCostmap)",26
5,Fallback(Sequence(Fallback)),26
6,Fallback(SequenceStar),24
7,"RecoveryNode(PipelineSequence(RateController, ...",23
8,"RecoveryNode(ComputePathToPose, ClearEntireCos...",21
9,"ReactiveFallback(GoalUpdated, SequenceStar(Cle...",21



___ Top 10 des Design Patterns Globaux (Toutes catégories) ___


,Pattern (Signature),Occurrences
0,SubTree,472
1,SetBlackboard,361
2,Sequence,326
3,ClearEntireCostmap,174
4,TextToSpeechActionClient,156
5,ComputePathToPose,143
6,FollowPath,136
7,SequenceStar,112
8,ForceFailure,93
9,Fallback,85


### Mini-conclusion Section 4

- Le motif Fallback(Sequence, Sequence) (28 occurrences) est le design pattern typique d'une mission sécurisée.
- Les motifs imbriqués Fallback(Sequence(Fallback)) (26 occurrences) vont permettre d'exposer notre modèle à des arbres profonds et des logiques de récupération en cascade.

- Beaucoup de SubTree, notre modèle va apprendre à concevoir des architectures modulaires. Le LLM pourra alors réutiliser des sous-missions de la SNCF.
- Le partage de variables (le "Blackboard") est l'action la plus courante avant même la navigation. Le modèle devra comprendre comment déclarer une variable et la passer de nœud en nœud.

- Les motifs de type Fallback/RecoveryNode sont récurrents et structurent la logique de résilience.
- Les signatures extraites capturent des schémas réutilisables de conception BT.
- Ces patterns peuvent servir de briques de génération et de validation pour le futur modèle.

## 5. Conclusion

- Dominance de la navigation spatiale : la majorité des arbres gèrent des déplacements (Nav2, waypoints, recovery), ce qui est cohérent avec NAV4RAIL et aligne bien le corpus avec notre cible.

- Qualité des données maîtrisée : sur 594 exemples initiaux, 23 XML invalides sont retirés, ce qui laisse 571 exemples exploitables pour l'entraînement (il est aussi possible de les corriger au besoin).

- Domaine d'exécution ouvert : le dataset reste très hétérogène avec 314 skills uniques (navigation, manipulation, vision, contrôle), ce qui favorise la généralisation syntaxique BT.CPP.

- Validation de la stratégie du "1er entonnoir" : si un LLM compact (7B) apprend correctement la syntaxe sur ce vocabulaire large, l'adaptation à un catalogue SNCF fermé (28 skills typés) devrait demander peu d'exemples supplémentaires.

## 6. Export JSONL pour Fine-Tuning Chat

### 6.1 Pourquoi ce format
Les modèles chat récents (LLaMA, Qwen, etc.) attendent un template strict avec rôles `system`, `user` et `assistant`.

In [47]:
output_file = DATASET_CHAT_PATH
with open(output_file, 'w', encoding='utf-8') as f_out:
    for item in dataset_for_jsonl:
        system_prompt = item.get("instruction", SYSTEM_PROMPT_DEFAULT)

        chat_format = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": item.get("input", "")},
                {"role": "assistant", "content": item.get("output", "")}
            ]
        }
        f_out.write(json.dumps(chat_format, ensure_ascii=False) + '\n')

print(f"___ Préparation terminée ___")
print(f"Fichier exporté pour TRL (QLoRA) : {output_file} ({len(dataset_for_jsonl)} exemples valides)")

___ Préparation terminée ___
Fichier exporté pour TRL (QLoRA) : bt_dataset_chat.jsonl (571 exemples valides)
